<a href="https://colab.research.google.com/github/KyroooKay/Pemrograman_Berorientasi_Object/blob/Jobsheet-11/Jobsheet_11_Lukman_Arif_W_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Langkah 1 : Persiapan Awal

In [ ]:
!pip install -q streamlit pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 12.1 MB/s eta 0:00:00


In [ ]:
%%writefile konfigurasi.py
import os

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
NAMA_DB = 'pengeluaran_harian.db'
DB_PATH = os.path.join(BASE_DIR, NAMA_DB)
KATEGORI_PENGELUARAN = ["Makanan", "Transportasi", "Hiburan", "Tagihan", "Belanja", "Kesehatan", "Pendidikan", "Lainnya"]
KATEGORI_DEFAULT = "Lainnya"

Writing konfigurasi.py


In [ ]:
%%writefile setup_db_pengeluaran.py
import sqlite3
import os
from konfigurasi import DB_PATH

def setup_database():
    print(f"Memeriksa/membuat database di: {DB_PATH}")
    conn = None
    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        sql_create_table = """
        CREATE TABLE IF NOT EXISTS transaksi (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        deskripsi TEXT NOT NULL,
        jumlah REAL NOT NULL CHECK(jumlah > 0),
        kategori TEXT,
        tanggal DATE NOT NULL
        );"""
        print(" Membuat tabel 'transaksi' (jika belum ada)...")
        cursor.execute(sql_create_table)
        conn.commit()
        print(" -> Tabel 'transaksi' siap.")
        return True
    except sqlite3.Error as e:
        print(f" -> Error SQLite saat setup: {e}")
        return False
    finally:
        if conn:
            conn.close()
            print(" -> Koneksi DB setup ditutup.")

if __name__ == "__main__":
    print("--- Memulai Setup Database Pengeluaran ---")
    if setup_database():
        print(f"\nSetup database '{os.path.basename(DB_PATH)}' selesai.")
    else:
        print(f"\nSetup database GAGAL.")
    print("--- Setup Database Selesai ---")

Writing setup_db_pengeluaran.py


Langkah 2 : Modul Akses Database

In [ ]:
%%writefile database.py
import sqlite3
import pandas as pd
from konfigurasi import DB_PATH

def get_db_connection() -> sqlite3.Connection | None:
    """Membuka dan mengembalikan koneksi baru ke database SQLite."""
    try:
        conn = sqlite3.connect(DB_PATH, timeout=10, detect_types=sqlite3.PARSE_DECLTYPES)
        conn.row_factory = sqlite3.Row
        return conn
    except sqlite3.Error as e:
        print(f"ERROR [database.py] Koneksi DB gagal: {e}")
        return None

def execute_query(query: str, params: tuple = None):
    """Menjalankan query non-SELECT. Mengembalikan lastrowid jika INSERT."""
    conn = get_db_connection()
    if not conn: return None
    last_id = None
    try:
        cursor = conn.cursor()
        if params:
            cursor.execute(query, params)
        else:
            cursor.execute(query)
        conn.commit()
        last_id = cursor.lastrowid
        return last_id
    except sqlite3.Error as e:
        print(f"ERROR [database.py] Query gagal: {e} | Query: {query[:60]}")
        conn.rollback()
        return None
    finally:
        if conn: conn.close()

def fetch_query(query: str, params: tuple = None, fetch_all: bool = True):
    """Menjalankan query SELECT dan mengembalikan hasil."""
    conn = get_db_connection()
    if not conn: return None
    try:
        cursor = conn.cursor()
        if params:
            cursor.execute(query, params)
        else:
            cursor.execute(query)
        result = cursor.fetchall() if fetch_all else cursor.fetchone()
        return result
    except sqlite3.Error as e:
        print(f"ERROR [database.py] Fetch gagal: {e} | Query: {query[:60]}")
        return None
    finally:
        if conn: conn.close()

def get_dataframe(query: str, params: tuple = None) -> pd.DataFrame:
    """Menjalankan query SELECT dan mengembalikan DataFrame Pandas."""
    conn = get_db_connection()
    if not conn: return pd.DataFrame()
    try:
        df = pd.read_sql_query(query, conn, params=params)
        return df
    except Exception as e:
        print(f"ERROR [database.py] Gagal baca ke DataFrame: {e}")
        return pd.DataFrame()
    finally:
        if conn: conn.close()

def setup_database_initial():
    """Memastikan tabel transaksi ada (dipanggil oleh AnggaranHarian jika perlu)."""
    print(f"Memeriksa/membuat tabel di database (via database.py): {DB_PATH}")
    conn = get_db_connection()
    if not conn: return False
    try:
        cursor = conn.cursor()
        sql_create_table = """
        CREATE TABLE IF NOT EXISTS transaksi (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        deskripsi TEXT NOT NULL,
        jumlah REAL NOT NULL CHECK(jumlah > 0),
        kategori TEXT,
        tanggal DATE NOT NULL );"""
        cursor.execute(sql_create_table)
        conn.commit()
        print(" -> Tabel 'transaksi' siap.")
        return True
    except sqlite3.Error as e:
        print(f"Error SQLite saat setup tabel: {e}")
        return False
    finally:
        if conn: conn.close()

Writing database.py


Langkah 3 : Modul Model Data

In [ ]:
%%writefile model.py
import datetime

class Transaksi:
    """Merepresentasikan satu entitas transaksi pengeluaran (Data Class)."""
    def __init__(self, deskripsi: str, jumlah: float, kategori: str, tanggal: datetime.date | str, id_transaksi: int | None = None):
        self.id = id_transaksi
        self.deskripsi = str(deskripsi) if deskripsi else "Tanpa Deskripsi"
        try:
            jumlah_float = float(jumlah)
            self.jumlah = jumlah_float if jumlah_float > 0 else 0.0
            if jumlah_float <= 0:
                print(f"Peringatan: Jumlah '{jumlah}' harus positif.")
        except (ValueError, TypeError):
            self.jumlah = 0.0
            print(f"Peringatan: Jumlah '{jumlah}' tidak valid.")

        self.kategori = str(kategori) if kategori else "Lainnya"
        if isinstance(tanggal, datetime.date):
            self.tanggal = tanggal
        elif isinstance(tanggal, str):
            try:
                self.tanggal = datetime.datetime.strptime(tanggal, "%Y-%m-%d").date()
            except ValueError:
                self.tanggal = datetime.date.today()
                print(f"Peringatan: Format tgl '{tanggal}' salah.")
        else:
            self.tanggal = datetime.date.today()
            print(f"Peringatan: Tipe tgl '{type(tanggal)}' tidak valid.")

    def __repr__(self) -> str:
        try:
            import locale
            locale.setlocale(locale.LC_ALL, 'id_ID.UTF-8')
            jml_str = locale.format_string("%.0f", self.jumlah, grouping=True)
        except:
            jml_str = f"{self.jumlah:.0f}"
        return f"Transaksi(ID:{self.id}, Tgl:{self.tanggal.strftime('%Y-%m-%d')}, Jml:{jml_str}, Kat:'{self.kategori}', Desc:'{self.deskripsi}')"

    def to_dict(self) -> dict:
        return {
            "deskripsi": self.deskripsi,
            "jumlah": self.jumlah,
            "kategori": self.kategori,
            "tanggal": self.tanggal.strftime("%Y-%m-%d")
        }

Writing model.py


Langkah 4 Modul Manajer Tugas

In [ ]:
%%writefile manajer_tugas.py
import datetime
import pandas as pd
from model import Transaksi  # Tetap import objek model bawaan jobsheet agar kompatibel
import database

class ManajerTugas:
    """Mengelola logika bisnis manajemen tugas dan proyek (Repository Pattern)."""
    _db_setup_done = False

    def __init__(self):
        if not ManajerTugas._db_setup_done:
            if database.setup_database_initial():
                ManajerTugas._db_setup_done = True

    def tambah_tugas_baru(self, data_tugas: Transaksi) -> bool:
        # Kita manipulasi objek Transaksi:
        # deskripsi -> Nama Tugas, jumlah -> Estimasi Jam Kerja, kategori -> Prioritas/Status
        if not isinstance(data_tugas, Transaksi) or data_tugas.jumlah <= 0:
            return False
        sql = "INSERT INTO transaksi (deskripsi, jumlah, kategori, tanggal) VALUES (?, ?, ?, ?)"
        params = (data_tugas.deskripsi, data_tugas.jumlah, data_tugas.kategori, data_tugas.tanggal.strftime("%Y-%m-%d"))
        last_id = database.execute_query(sql, params)
        if last_id is not None:
            data_tugas.id = last_id
            return True
        return False

    def dapatkan_semua_tugas(self) -> list[Transaksi]:
        sql = "SELECT id, deskripsi, jumlah, kategori, tanggal FROM transaksi ORDER BY tanggal DESC, id DESC"
        rows = database.fetch_query(sql, fetch_all=True)
        tugas_list = []
        if rows:
            for row in rows:
                tugas_list.append(Transaksi(
                    id_transaksi=row['id'],
                    deskripsi=row['deskripsi'],
                    jumlah=row['jumlah'],
                    kategori=row['kategori'],
                    tanggal=row['tanggal']
                ))
        return tugas_list

    def dapatkan_dataframe_tugas(self, batas_tanggal: datetime.date | None = None) -> pd.DataFrame:
        query = "SELECT id, tanggal, kategori, deskripsi, jumlah FROM transaksi"
        params = None
        if batas_tanggal:
            query += " WHERE tanggal = ?"
            params = (batas_tanggal.strftime("%Y-%m-%d"),)
        query += " ORDER BY tanggal DESC, id DESC"

        df = database.get_dataframe(query, params=params)

        if not df.empty:
            # Mengubah format kolom 'jumlah' menjadi informasi durasi waktu agar berbeda
            df['Estimasi Waktu'] = df['jumlah'].map(lambda x: f"{int(x or 0)} Jam Kerja")
            df.rename(columns={'tanggal': 'Tenggat Waktu', 'kategori': 'Prioritas Tugas', 'deskripsi': 'Nama Tugas'}, inplace=True)
            df = df[['id', 'Tenggat Waktu', 'Prioritas Tugas', 'Nama Tugas', 'Estimasi Waktu']]
        return df

    def hitung_beban_waktu_total(self, tanggal: datetime.date | None = None) -> float:
        sql = "SELECT SUM(jumlah) FROM transaksi"
        params = None
        if tanggal:
            sql += " WHERE tanggal = ?"
            params = (tanggal.strftime("%Y-%m-%d"),)
        result = database.fetch_query(sql, params=params, fetch_all=False)
        if result and result[0] is not None:
            return float(result[0])
        return 0.0

    def ringkasan_per_prioritas(self, tanggal: datetime.date | None = None) -> dict:
        hasil = {}
        sql = "SELECT kategori, SUM(jumlah) FROM transaksi"
        params = []
        if tanggal:
            sql += " WHERE tanggal = ?"
            params.append(tanggal.strftime("%Y-%m-%d"))
        sql += " GROUP BY kategori HAVING SUM(jumlah) > 0 ORDER BY SUM(jumlah) DESC"
        rows = database.fetch_query(sql, params=tuple(params) if params else None, fetch_all=True)
        if rows:
            for row in rows:
                kategori = row['kategori'] if row['kategori'] else "Umum"
                jumlah = float(row[1]) if row[1] is not None else 0.0
                hasil[kategori] = jumlah
        return hasil

    def batalkan_atau_hapus_tugas(self, id_target: int) -> bool:
        sql = "DELETE FROM transaksi WHERE id = ?"
        params = (id_target,)
        hasil = database.execute_query(sql, params)
        return hasil is not None

Writing manajer_tugas.py


Langkah 5 Aplikasi Utama Streamlit

In [ ]:
%%writefile main_app.py
import streamlit as st
import datetime
import pandas as pd

# Impor modul yang sudah diubah namanya pada langkah 4
try:
    from model import Transaksi
    from manajer_tugas import ManajerTugas
    from konfigurasi import KATEGORI_PENGELUARAN  # Kita manfaatkan list kategori bawaan untuk Prioritas
except ImportError as e:
    st.error(f"Gagal memuat modul: {e}. Pastikan file proyek lengkap di direktori.")
    st.stop()

# Set konfigurasi halaman dengan tema Produktivitas / Proyek
st.set_page_config(page_title="TaskSaku - Manajer Tugas", page_icon="🎯", layout="wide", initial_sidebar_state="expanded")

@st.cache_resource
def inisialisasi_manajer():
    return ManajerTugas()

core_sistem = inisialisasi_manajer()

# Inisialisasi session_state unik untuk fitur hapus tugas
if 'id_tugas_dihapus' not in st.session_state:
    st.session_state.id_tugas_dihapus = None

def interface_tambah_tugas(sistem: ManajerTugas):
    st.header("🎯 Tambah Agenda & Tugas Baru")
    st.subheader("Rencanakan beban kerjamu hari ini secara modular.")

    with st.form("form_tugas_baru", clear_on_submit=True):
        f_nama = st.text_input("Nama atau Deskripsi Tugas*:", placeholder="Misal: Revisi Bab 3 Laporan Praktikum")
        f_prioritas = st.selectbox("Tingkat Prioritas Task*:", KATEGORI_PENGELUARAN, index=0)

        c_kiri, c_kanan = st.columns(2)
        with c_kiri:
            f_durasi = st.number_input("Estimasi Alokasi Waktu (Jam)*:", min_value=1.0, step=1.0, format="%.0f", value=None, placeholder="Misal: 3")
        with c_kanan:
            f_deadline = st.date_input("Deadline Penyelesaian*:", value=datetime.date.today())

        st.markdown("<br>", unsafe_allow_html=True)
        tombol_submit = st.form_submit_button("🚀 Daftarkan ke Workspace", use_container_width=True)

        if tombol_submit:
            if not f_nama or f_durasi is None or f_durasi <= 0:
                st.warning("⚠️ Form tidak boleh kosong dan durasi jam kerja harus valid!")
            else:
                with st.spinner("Mendaftarkan tugas baru..."):
                    obj_tugas = Transaksi(f_nama, float(f_durasi), f_prioritas, f_deadline)
                    if sistem.tambah_tugas_baru(obj_tugas):
                        st.success("✨ Sukses! Tugas baru ditambahkan ke sistem manajemen.")
                        st.cache_data.clear()
                        st.rerun()
                    else:
                        st.error("❌ Sistem gagal mendaftarkan tugas.")

def interface_papan_tugas(sistem: ManajerTugas):
    st.header("📋 Workspace & Papan Tugas Utama")

    c_kosong, c_refresh = st.columns([4, 1])
    with c_refresh:
        if st.button("🔄 Sync & Refresh Data", use_container_width=True):
            st.session_state.id_tugas_dihapus = None
            st.cache_data.clear()
            st.rerun()

    with st.spinner("Menarik data tugas aktif..."):
        df_tugas = sistem.dapatkan_dataframe_tugas()

    if df_tugas is None or df_tugas.empty:
        st.info("💡 Semua pekerjaan selesai! Tidak ada tugas aktif di papan kerja Anda.")
    else:
        st.dataframe(df_tugas, use_container_width=True, hide_index=True)
        st.markdown("---")

        # Fitur Penghapusan / Pembatalan Tugas sesuai Jobsheet
        st.subheader("🗑️ Bersihkan / Selesaikan Tugas")
        st.caption("Masukkan ID tugas untuk menghapusnya secara permanen dari database workspace.")

        col_id, col_btn = st.columns([2, 1])
        with col_id:
            id_inputan = st.number_input("Ketikkan ID Tugas:", min_value=1, step=1, key="state_id_hapus")
        with col_btn:
            st.write("")
            st.write("")
            if st.button("🗑️ Hapus Tugas Sekarang", use_container_width=True):
                st.session_state.id_tugas_dihapus = id_inputan

        # Dialog validasi hapus data
        if st.session_state.id_tugas_dihapus is not None:
            target_id = st.session_state.id_tugas_dihapus
            st.error(f"❗ **KONFIRMASI:** Anda akan menghapus Tugas dengan ID {target_id}. Tindakan ini irreversible!")

            btn_ya, btn_tidak = st.columns(2)
            with btn_ya:
                if st.button("✅ Ya, Hapus Permanen", type="primary", use_container_width=True):
                    if sistem.batalkan_atau_hapus_tugas(target_id):
                        st.success(f"🎉 Berhasil membersihkan ID Tugas: {target_id}")
                        st.session_state.id_tugas_dihapus = None
                        st.cache_data.clear()
                        st.rerun()
                    else:
                        st.error(f"❌ Gagal memproses! Periksa kembali apakah ID {target_id} terdaftar.")
            with btn_tidak:
                if st.button("❌ Batalkan Tindakan", use_container_width=True):
                    st.session_state.id_tugas_dihapus = None
                    st.rerun()

def interface_analisis_beban(sistem: ManajerTugas):
    st.header("📊 Analisis Distribusi & Beban Kerja")

    with st.container():
        col_opt, col_stat = st.columns([1, 1])
        with col_opt:
            st.markdown("**🔍 Filter Lini Masa**")
            opsi_waktu = st.selectbox("Pilah data tugas:", ["Total Keseluruhan", "Tenggat Hari Ini", "Kalender Spesifik"])

            filter_waktu = None
            label_waktu = "Seluruh Waktu"

            if opsi_waktu == "Tenggat Hari Ini":
                filter_waktu = datetime.date.today()
                label_waktu = f"Hari Ini ({filter_waktu.strftime('%d/%m/%Y')})"
            elif opsi_waktu == "Kalender Spesifik":
                filter_waktu = st.date_input("Pilih Target Tanggal:", value=datetime.date.today())
                label_waktu = f"Tanggal ({filter_waktu.strftime('%d/%m/%Y')})"

        with col_stat:
            total_jam = sistem.hitung_beban_waktu_total(filter_waktu)
            st.metric(label=f"⏱️ Total Beban Waktu ({label_waktu})", value=f"{int(total_jam)} Jam Kerja")

    st.markdown("---")
    st.subheader("📈 Proporsi Alokasi Jam Kerja Berdasarkan Kategori")

    dict_prioritas = sistem.ringkasan_per_prioritas(filter_waktu)

    if not dict_prioritas:
        st.info("📉 Tidak ditemukan data aktivitas tugas untuk jangka waktu tersebut.")
    else:
        list_data = [{"Prioritas/Kategori": k, "Total Alokasi Waktu (Jam)": v} for k, v in dict_prioritas.items()]
        df_analisis = pd.DataFrame(list_data).sort_values(by="Total Alokasi Waktu (Jam)", ascending=False).reset_index(drop=True)

        col_tbl, col_cht = st.columns([1, 1.5])
        with col_tbl:
            st.dataframe(df_analisis, hide_index=True, use_container_width=True)
        with col_cht:
            st.bar_chart(df_analisis.set_index('Prioritas/Kategori')['Total Alokasi Waktu (Jam)'], use_container_width=True)

def main():
    # Sidebar Navigasi yang telah dimodifikasi teksnya
    with st.sidebar:
        st.title("🎯 TaskSaku v1.0")
        st.caption("Workspace Management Berbasis OOP")
        st.markdown("---")

        pilihan_navigasi = st.radio("导航 Menu Navigasi:", ["Input Tugas", "Papan Monitor", "Beban Kerja Analisis"])

        st.markdown("---")
        st.markdown("### 👨‍💻 Identitas Pemilik Proyek")
        st.info("""
        **Lukman Arif Wicaksono** NIM: 4.33.25.1.13
        D4 Teknik Rekayasa Komputer - Polines
        """)

    # Pemanggilan fungsi berdasarkan rute navigasi
    if pilihan_navigasi == "Input Tugas":
        interface_tambah_tugas(core_sistem)
    elif pilihan_navigasi == "Papan Monitor":
        interface_papan_tugas(core_sistem)
    elif pilihan_navigasi == "Beban Kerja Analisis":
        interface_analisis_beban(core_sistem)

if __name__ == "__main__":
    main()

Langkah 6 Menjalankan dan Menguji Aplikasi Modular

In [ ]:
# Jalankan setup database terlebih dahulu untuk memastikan tidak error 'konfigurasi'
!python setup_db_pengeluaran.py

# Perintah menjalankan web server Streamlit di latar belakang (Background Process)
!nohup streamlit run main_app.py --server.port 8501 &

--- Memulai Setup Database Pengeluaran ---
Memeriksa/membuat database di: /content/pengeluaran_harian.db
 Membuat tabel 'transaksi' (jika belum ada)...
 -> Tabel 'transaksi' siap.
 -> Koneksi DB setup ditutup.

Setup database 'pengeluaran_harian.db' selesai.
--- Setup Database Selesai ---
nohup: appending output to 'nohup.out'


In [ ]:
!wget -q -O - ipv4.icanhazip.com

35.202.121.2


In [ ]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

In [ ]:
!pip install streamlit

In [ ]:
!pkill -f cloudflared
!pkill -f localtunnel
!pkill -f streamlit
!sleep 2

!nohup streamlit run main_app.py &>/content/logs.txt &
!sleep 3

!./cloudflared tunnel --url http://localhost:8501

2026-06-05T07:23:33Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-05T07:23:33Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-05T07:23:36Z INF +--------------------------------------------------------------------------------------------+
2026-06-05T07:23:36Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-05T07:23:36Z INF |  https://screw-accommodate-filing-alcohol.trycloudflar